# **PubMed Data Mining Pipeline Overview**

In [ ]:
#!/usr/bin/env python3
"""
PubMed miner (broad): Plants / phytochemicals for psoriasis
- Robust XML parsing using ElementTree (no regex parsing of XML)
- Extracts: title, abstract, year, journal, authors, mesh terms, substances
- Adds best-effort extraction:
    1) Plant_Scientific_Names (binomial)
    2) Plant_Common_Names (heuristic)
    3) Phytochemical_Names (substances + heuristic)
    4) Gene_Symbols (HGNC-like symbols from title/abstract)
    5) Where_Found_Source (sentences: "isolated from", "derived from", etc.)
    6) Indian_Name (vernacular / Hindi name if mentioned OR via small mapping)
    7) Psoriasis_Type (if detected, else "Psoriasis")
- Exports CSV + JSON
"""

import time, csv, json, re
from typing import List, Dict, Optional, Tuple
import requests
import xml.etree.ElementTree as ET


## **Configuration and API Settings**

In [ ]:
# 1) SETTINGS
EMAIL = "your_email@example.com"   # recommended by NCBI
API_KEY = ""                      # optional
RETMAX = 300
SLEEP_SEC = 0.34 if API_KEY else 0.8

OUT_CSV = "pubmed_psoriasis_plants_phytochemicals_enriched.csv"
OUT_JSON = "pubmed_psoriasis_plants_phytochemicals_enriched.json"

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

## **Query Construction for Plant–Psoriasis Literature Retrieval**

In [ ]:
# 2) QUERY (BROAD)
def build_broad_query() -> str:
    psoriasis_block = '(psoriasis[Title/Abstract] OR psoriatic[Title/Abstract])'

    plants_block = (
        '(plant[Title/Abstract] OR plants[Title/Abstract] OR herbal[Title/Abstract] OR '
        '"medicinal plant"[Title/Abstract] OR "plant extract"[Title/Abstract] OR extract*[Title/Abstract] OR '
        'phytochemical*[Title/Abstract] OR "natural product"[Title/Abstract] OR '
        'polyphenol*[Title/Abstract] OR flavonoid*[Title/Abstract] OR alkaloid*[Title/Abstract] OR terpen*[Title/Abstract])'
    )

    india_boost = (
        '(India[Title/Abstract] OR Indian[Title/Abstract] OR Ayurveda[Title/Abstract] OR Ayurvedic[Title/Abstract] '
        'OR "traditional medicine"[Title/Abstract] OR ethnobotan*[Title/Abstract])'
    )

    topical_boost = '(skin[Title/Abstract] OR topical[Title/Abstract] OR dermatitis[Title/Abstract] OR inflammation[Title/Abstract])'

    return f"{psoriasis_block} AND {plants_block} AND ({india_boost} OR {topical_boost})"


## **PubMed E-Utilities Data Retrieval Functions**

In [ ]:
# 3) E-UTILS
def esearch(query: str, retmax: int = 200) -> List[str]:
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
        "email": EMAIL
    }
    if API_KEY:
        params["api_key"] = API_KEY

    r = requests.get(f"{EUTILS}/esearch.fcgi", params=params, timeout=60)
    r.raise_for_status()
    return r.json().get("esearchresult", {}).get("idlist", [])


def efetch_xml(pmids: List[str]) -> str:
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "email": EMAIL
    }
    if API_KEY:
        params["api_key"] = API_KEY

    r = requests.get(f"{EUTILS}/efetch.fcgi", params=params, timeout=60)
    r.raise_for_status()
    return r.text

## **Text Processing and Feature Extraction Utilities**

In [ ]:
# 4) HELPERS
PSORIASIS_TYPE_PATTERNS = [
    ("Plaque Psoriasis", r"\b(plaque psoriasis|psoriasis vulgaris|vulgaris)\b"),
    ("Guttate Psoriasis", r"\b(guttate)\b"),
    ("Inverse Psoriasis", r"\b(inverse psoriasis|flexural psoriasis)\b"),
    ("Pustular Psoriasis", r"\b(pustular)\b"),
    ("Erythrodermic Psoriasis", r"\b(erythrodermic)\b"),
    ("Scalp Psoriasis", r"\b(scalp psoriasis)\b"),
    ("Nail Psoriasis", r"\b(nail psoriasis)\b"),
    ("Psoriatic Arthritis", r"\b(psoriatic arthritis)\b"),
]

# Binomial scientific names: Genus species
BINOMIAL_REGEX = re.compile(r"\b([A-Z][a-z]{2,})\s([a-z]{3,})\b")

# Heuristic phytochemical words (expand anytime)
PHYTOCHEM_HINTS = re.compile(
    r"\b(curcumin|resveratrol|quercetin|apigenin|kaempferol|catechin|epigallocatechin|"
    r"berberine|glycyrrhizin|piperine|allicin|thymoquinone|cinnamaldehyde|"
    r"azadirachtin|nimbolide|mahanine|withaferin|boswellic|gingerol|eugenol|"
    r"ursolic acid|oleanolic acid|lupeol|beta-sitosterol|stigmasterol|"
    r"tannic acid|gallic acid|ellagic acid)\b",
    flags=re.IGNORECASE
)

# Gene symbol extraction:
# - typically all caps, may include digits/hyphen
# - avoid very short tokens (like "A", "II") and common false positives
GENE_REGEX = re.compile(r"\b[A-Z0-9]{2,10}(?:-[A-Z0-9]{1,5})?\b")

GENE_STOP = {
    # common non-genes in abstracts
    "DNA","RNA","PCR","ELISA","IL","TNF",  # note: IL and TNF can be genes, but IL alone often noise
    "NF","MAPK","STAT","ROS","NO","UV","UVB","UVA",
    "WHO","USA","UK","FIG","FIGS","TABLE","P","CI","OR",
    "HPLC","LCMS","MS","NMR","IC50","EC50","SD","SEM",
    "PSO","PASI","DLQI",
    "MEDLINE","PUBMED",
}

# Sentences that may mention source/where found
SOURCE_SENTENCE_PAT = re.compile(
    r"\b(found in|isolated from|derived from|extracted from|obtained from|from the leaves of|from the bark of|"
    r"from the roots of|from the seeds of|from rhizome|from flower|from fruit|from peel)\b",
    flags=re.IGNORECASE
)

# Vernacular/hindi name patterns (best-effort)
VERNACULAR_PAT = re.compile(
    r"\b(hindi|urdu|tamil|telugu|marathi|bengali|gujarati|punjabi|kannada|malayalam)\s*[:\-]\s*([A-Za-z \-]{2,40})",
    flags=re.IGNORECASE
)

# Small editable mapping for Indian household plants/ingredients -> Indian name(s)
# Add more as you like.
INDIAN_NAME_MAP = {
    "Azadirachta indica": "Neem / Nim",
    "Curcuma longa": "Haldi (Turmeric)",
    "Withania somnifera": "Ashwagandha",
    "Ocimum tenuiflorum": "Tulsi",
    "Tinospora cordifolia": "Giloy",
    "Zingiber officinale": "Adrak (Ginger)",
    "Allium sativum": "Lahsun (Garlic)",
    "Piper nigrum": "Kali Mirch (Black pepper)",
    "Cinnamomum verum": "Dalchini (Cinnamon)",
    "Nigella sativa": "Kalonji",
}

COMMON_PLANT_WORDS = re.compile(
    r"\b(neem|turmeric|haldi|tulsi|ashwagandha|giloy|ginger|garlic|black pepper|pepper|"
    r"cinnamon|kalonji|aloe vera|amla|basil|curry leaf|curry leaves|methi|fenugreek)\b",
    flags=re.IGNORECASE
)

def split_sentences(text: str) -> List[str]:
    # simple sentence splitter (good enough for abstracts)
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def guess_psoriasis_type(title: str, abstract: str, mesh_terms: List[str]) -> str:
    text = f"{title} {abstract}".lower()
    for label, pat in PSORIASIS_TYPE_PATTERNS:
        if re.search(pat, text, flags=re.IGNORECASE):
            return label
    return "Psoriasis"

def extract_scientific_plants(title: str, abstract: str) -> List[str]:
    text = f"{title} {abstract}"
    pairs = BINOMIAL_REGEX.findall(text)
    bad_second = {"states", "kingdom"}
    out = []
    for g, s in pairs:
        if s.lower() in bad_second:
            continue
        # avoid obvious non-biological phrases
        if g.lower() in {"united"}:
            continue
        out.append(f"{g} {s}")
    # de-duplicate while preserving order
    seen = set()
    uniq = []
    for x in out:
        if x not in seen:
            uniq.append(x)
            seen.add(x)
    return uniq

def extract_common_plants(title: str, abstract: str) -> List[str]:
    text = f"{title} {abstract}"
    hits = COMMON_PLANT_WORDS.findall(text)
    # normalize
    norm = []
    for h in hits:
        norm.append(h.lower().strip())
    seen = set()
    uniq = []
    for x in norm:
        if x not in seen:
            uniq.append(x)
            seen.add(x)
    return uniq

def extract_phytochemicals(title: str, abstract: str, substances: List[str]) -> List[str]:
    """
    Sources:
    1) PubMed ChemicalList substances (often good for chemicals)
    2) Known phytochemical hint list in text
    """
    text = f"{title} {abstract}"

    ph = set()
    # substances: keep those that look like chemicals (not perfect)
    for s in substances:
        if not s:
            continue
        # crude filter: avoid generic terms like "Plant Extracts"
        if s.lower() in {"plant extracts", "phytotherapy", "medicine, ayurvedic"}:
            continue
        ph.add(s.strip())

    for m in PHYTOCHEM_HINTS.findall(text):
        ph.add(m.strip())

    # Keep stable-ish, short list
    return sorted(ph)

def extract_genes(title: str, abstract: str) -> List[str]:
    text = f"{title} {abstract}"
    tokens = GENE_REGEX.findall(text)
    genes = []
    for t in tokens:
        if len(t) < 2:
            continue
        if t in GENE_STOP:
            continue
        # discard all-digit tokens
        if t.isdigit():
            continue
        # discard common units/notations
        if re.fullmatch(r"\d+(\.\d+)?", t):
            continue
        # very common false positives in biomedical abstracts
        if t in {"IL-6","IL-17","IL-23","TNF-α","TNF-a","NF-κB"}:
            # These are meaningful; keep
            genes.append(t)
            continue
        # If token is mixed letters/digits like "H1N1" -> likely not a gene in this context
        if re.fullmatch(r"[A-Z]{1,2}\d{2,}", t):
            continue
        # Many real genes are like STAT3, IL17A, TNF, IFNG, etc.
        genes.append(t)

    # de-duplicate
    seen = set()
    uniq = []
    for g in genes:
        if g not in seen:
            uniq.append(g)
            seen.add(g)
    return uniq

def extract_where_found(title: str, abstract: str) -> str:
    text = f"{title} {abstract}".strip()
    if not text:
        return ""
    sents = split_sentences(text)
    hits = [s for s in sents if SOURCE_SENTENCE_PAT.search(s)]
    # return up to 2 most informative
    return " | ".join(hits[:2])

def extract_indian_name(scientific_plants: List[str], title: str, abstract: str) -> str:
    """
    Best-effort:
    1) If abstract explicitly mentions language: "Hindi: ..."
    2) Else map from scientific names via INDIAN_NAME_MAP (if present)
    """
    text = f"{title} {abstract}"
    m = VERNACULAR_PAT.search(text)
    if m:
        # e.g., Hindi: Neem
        lang = m.group(1).strip().title()
        name = m.group(2).strip()
        return f"{lang}: {name}"

    for sp in scientific_plants:
        if sp in INDIAN_NAME_MAP:
            return INDIAN_NAME_MAP[sp]

    return ""

## **XML Parsing and Record Structuring**

In [ ]:
# 5) XML PARSER
def get_text(elem: Optional[ET.Element]) -> str:
    return (elem.text or "").strip() if elem is not None else ""

def parse_pubmed_xml(xml_text: str) -> List[Dict]:
    root = ET.fromstring(xml_text)
    records = []

    for art in root.findall(".//PubmedArticle"):
        pmid = get_text(art.find(".//MedlineCitation/PMID"))

        title = get_text(art.find(".//Article/ArticleTitle"))

        abstract_parts = []
        for ab in art.findall(".//Article/Abstract/AbstractText"):
            abstract_parts.append("".join(ab.itertext()).strip())
        abstract = " ".join([p for p in abstract_parts if p]).strip()

        journal = get_text(art.find(".//Article/Journal/Title"))

        year = ""
        year_elem = art.find(".//Article/Journal/JournalIssue/PubDate/Year")
        if year_elem is not None and year_elem.text:
            year = year_elem.text.strip()
        else:
            md = art.find(".//Article/Journal/JournalIssue/PubDate/MedlineDate")
            if md is not None and md.text:
                m = re.search(r"(19|20)\d{2}", md.text)
                year = m.group(0) if m else ""

        authors = []
        for au in art.findall(".//Article/AuthorList/Author"):
            ln = get_text(au.find("LastName"))
            ini = get_text(au.find("Initials"))
            if ln:
                authors.append(f"{ln} {ini}".strip())
        authors_str = ", ".join(authors[:15]) + (" et al." if len(authors) > 15 else "")

        mesh_terms = []
        for mh in art.findall(".//MedlineCitation/MeshHeadingList/MeshHeading/DescriptorName"):
            t = get_text(mh)
            if t:
                mesh_terms.append(t)
        mesh_terms = sorted(set(mesh_terms))

        substances = []
        for sub in art.findall(".//MedlineCitation/ChemicalList/Chemical/NameOfSubstance"):
            t = get_text(sub)
            if t:
                substances.append(t)
        substances = sorted(set(substances))

        # Enriched fields
        psoriasis_type = guess_psoriasis_type(title, abstract, mesh_terms)

        sci_plants = extract_scientific_plants(title, abstract)
        common_plants = extract_common_plants(title, abstract)

        phytochemicals = extract_phytochemicals(title, abstract, substances)
        genes = extract_genes(title, abstract)

        where_found = extract_where_found(title, abstract)
        indian_name = extract_indian_name(sci_plants, title, abstract)

        records.append({
            "PMID": pmid,
            "Year": year,
            "Title": title,
            "Journal": journal,
            "Authors": authors_str,
            "Psoriasis_Type": psoriasis_type if psoriasis_type else "Psoriasis",
            "Plant_Scientific_Names": "; ".join(sci_plants),
            "Plant_Common_Names": "; ".join(common_plants),
            "Phytochemical_Names": "; ".join(phytochemicals),
            "Gene_Symbols": "; ".join(genes),
            "Where_Found_Source": where_found,
            "Indian_Name": indian_name,
            "Substances": "; ".join(substances),
            "MeSH_Terms": "; ".join(mesh_terms),
            "Abstract": abstract,
        })

    return records

## **Automated Literature Mining Workflow Execution**

In [ ]:
# 6) RUN
def main():
    query = build_broad_query()
    print("\nQUERY:\n", query)

    pmids = esearch(query, retmax=RETMAX)
    print(f"\nFound {len(pmids)} PMIDs (top by relevance).")

    if not pmids:
        print("\nNo PMIDs returned. Relax query blocks.")
        return

    all_records = []
    seen = set()

    batch_size = 50
    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i+batch_size]
        time.sleep(SLEEP_SEC)
        xml = efetch_xml(batch)
        recs = parse_pubmed_xml(xml)

        for r in recs:
            if not r.get("PMID") or r["PMID"] in seen:
                continue
            r["Query_Used"] = query
            seen.add(r["PMID"])
            all_records.append(r)

    if not all_records:
        print("\nGot PMIDs but no parsed records. (Unexpected.)")
        return

    cols = [
        "PMID", "Year", "Title", "Journal", "Authors",
        "Psoriasis_Type",
        "Plant_Scientific_Names", "Plant_Common_Names",
        "Phytochemical_Names",
        "Gene_Symbols",
        "Where_Found_Source",
        "Indian_Name",
        "Substances", "MeSH_Terms", "Abstract",
        "Query_Used"
    ]

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in all_records:
            w.writerow({c: r.get(c, "") for c in cols})

    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_records, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Saved CSV: {OUT_CSV}")
    print(f"✅ Saved JSON: {OUT_JSON}")
    print(f"Total unique papers: {len(all_records)}")


if __name__ == "__main__":
    main()

# **FILE PREPARATION FOR SYNERGY ANALYSIS**

In [ ]:
# ==============================================
# SynergyFinder 2-drug file (example-style) — YOUR PHYTOCHEMICALS
# Creates a pairwise-only CSV in the same "PairIndex numeric" style.
#
# IMPORTANT:
# - This generates the INPUT FILE for SynergyFinder. You still need to
#   paste/upload it into SynergyFinder.
# - SynergyFinder needs "Response" values from experiments (cell viability, etc.).
#   If you don't have them yet, set SIMULATE=False (Response blank) OR True to test.
# ==============================================

import os, numpy as np, pandas as pd

OUTDIR = "/content/sf_pairwise_phytochemicals"
os.makedirs(OUTDIR, exist_ok=True)

# ---- Put your phytochemicals in order (Drug1..DrugN) ----
# Keep names EXACTLY as you want to appear in SynergyFinder.
DRUGS = [
    "Curcumin",
    "Nimbolide",
    "Aloe gel polysaccharides",
    "Piperine",
    "Thymoquinone",
    "Giloy",                  # (Tinospora cordifolia extract/actives)
    "Withanolides",           # (Ashwagandha actives)
    "Asiaticoside",
    "Madecassoside",
    "Boswellic acids",
    "Glycyrrhizin",
    "Eugenol",
    "Ursolic acid",
    "Vitamin C",
    "Polyphenols"
]

# ---- Choose which pairs you want SynergyFinder to compute ----
# Option A (RECOMMENDED): all-vs-mahanine is not here, so choose a hub compound.
# If your "novel" is mahanine, add it to DRUGS and set HUB="Mahanine".
# For your current list, pick HUB = "Curcumin" (or "Nimbolide") for a manageable set.
HUB = "Curcumin"

# Build pairs: HUB with everyone else (PairIndex = 1..K)
pair_list = [(HUB, d) for d in DRUGS if d != HUB]

# Option B: uncomment to generate ALL possible pairs (can become huge)
# from itertools import combinations
# pair_list = list(combinations(DRUGS, 2))

# Map to PairIndex numeric style
PAIR_PLAN = {i+1: pair for i, pair in enumerate(pair_list)}

## **Concentration Grid Design and Response Simulation**

In [ ]:
# ---- Concentration grid ----
# Use realistic ranges. For phytochemicals, µM is common.
# SynergyFinder accepts any consistent unit label; keep it consistent across file.
GRID = [0, 0.1, 0.3, 1, 3, 10, 30, 100]   # µM
UNIT = "uM"

# ---- Response ----
# Set SIMULATE=True only to test SynergyFinder import.
# For real synergy, Response must be your measured endpoint:
# e.g., % inhibition, % viability, normalized response, etc.
SIMULATE = True
np.random.seed(11)

# Simple simulation (Bliss-like) just for testing imports
EMAX = {d: 100 for d in DRUGS}
EC50 = {d: np.random.uniform(1, 30) for d in DRUGS}   # random EC50 in µM for simulation only
HILL = {d: 1.2 for d in DRUGS}

def emax_model(c, d):
    if c <= 0:
        return 0.0
    return EMAX[d] * (c**HILL[d])/(EC50[d]**HILL[d] + c**HILL[d])

def sim_pair(c1, d1, c2, d2):
    e1, e2 = emax_model(c1,d1), emax_model(c2,d2)
    bliss = e1 + e2 - (e1*e2/100.0)
    return float(np.clip(bliss + np.random.normal(0, 2.0), -20, 110))

# ---- Build CSV rows ----
rows = []
for pidx, (d1, d2) in PAIR_PLAN.items():
    for c1 in GRID:
        for c2 in GRID:
            rows.append({
                "PairIndex": pidx,
                "Drug1": d1,
                "Drug2": d2,
                "Conc1": c1,
                "Conc2": c2,
                "Response": sim_pair(c1,d1,c2,d2) if SIMULATE else "",
                "ConcUnit": UNIT
            })

df = pd.DataFrame(rows, columns=["PairIndex","Drug1","Drug2","Conc1","Conc2","Response","ConcUnit"])

out_csv = f"{OUTDIR}/SF_PAIRWISE_{HUB}_vs_all_{UNIT}.csv"
df.to_csv(out_csv, index=False)

print("Created:", out_csv)
print("Total pairs:", len(PAIR_PLAN))
display(df.head(12))

# Also save the PairIndex legend so you don’t forget which pair is which
legend = pd.DataFrame(
    [{"PairIndex": k, "Drug1": v[0], "Drug2": v[1]} for k, v in PAIR_PLAN.items()],
    columns=["PairIndex","Drug1","Drug2"]
)
legend_path = f"{OUTDIR}/PAIRINDEX_LEGEND.csv"
legend.to_csv(legend_path, index=False)
print("Pair legend:", legend_path)
display(legend.head(20))